# Real-Time Agriculture Analytics using India Open Government Data

This notebook executes the complete project pipeline for agricultural data cleaning, statistical analysis, and yield prediction. It uses the cleaned dataset generated by `data_pipeline.py` and creates visual analytics that can be used for Tableau-style storytelling and reporting.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from data_pipeline import run_pipeline

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

print("Running agriculture analytics pipeline...")
df = run_pipeline()
print(f"Cleaned dataset shape: {df.shape}")
df.head()

## 1. Data Overview

The raw Agmarknet data is cleaned by fixing column names, removing duplicates, filling missing numeric values with the median within each crop, and engineering new features such as yield and revenue. The processed output is saved to `data/cleaned_agriculture_data.csv` for use in dashboarding and analysis.


In [ ]:
print('Dataset Statistics:')
print(df[['Area', 'Production', 'Yield', 'Modal_Price', 'Estimated_Revenue_INR']].describe().T)
print(f"\nUnique Crops: {df['Crop'].nunique()}")
print(f"Unique Seasons: {df['Season'].nunique()}")
print(f"Missing values:\n{df.isnull().sum()}")


## 2. Correlation Analysis

Pearson correlation helps reveal whether yield, production, and modal price move together. This supports interpretation of pricing dynamics and crop performance.


In [ ]:
corr_cols = ['Area', 'Production', 'Yield', 'Modal_Price']
corr_matrix = df[corr_cols].corr(method='pearson')
print('Pearson Correlation Matrix:')
print(corr_matrix)

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5, cbar_kws={'label': 'Correlation'})
plt.title('Pearson Correlation Heatmap: Agricultural Metrics')
plt.tight_layout()
plt.show()

## 3. Yield vs Revenue Scatter Plot

This visualization explores whether higher crop yield is associated with stronger market revenue potential across different seasons.


In [ ]:
plt.figure(figsize=(10, 7))
sns.scatterplot(data=df, x='Yield', y='Estimated_Revenue_INR', hue='Season', alpha=0.7, s=100, palette='husl')
plt.title('Yield vs Estimated Revenue by Season', fontsize=14, fontweight='bold')
plt.xlabel('Yield (Production/Area)', fontsize=12)
plt.ylabel('Estimated Revenue (INR)', fontsize=12)
plt.legend(title='Season', loc='best')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Production and Yield by Crop

Comparing top crops in terms of production volume and yield efficiency.


In [ ]:
top_crops_production = df.groupby('Crop')['Production'].sum().nlargest(10)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

top_crops_production.sort_values().plot(kind='barh', ax=ax1, color='skyblue')
ax1.set_title('Top 10 Crops by Total Production', fontweight='bold')
ax1.set_xlabel('Total Production')

top_crops_yield = df.groupby('Crop')['Yield'].mean().nlargest(10)
top_crops_yield.sort_values().plot(kind='barh', ax=ax2, color='lightcoral')
ax2.set_title('Top 10 Crops by Average Yield', fontweight='bold')
ax2.set_xlabel('Average Yield')

plt.tight_layout()
plt.show()

## 5. Seasonal Price Trends

Understanding how modal prices (market prices) vary across seasons.


In [ ]:
seasonal_price = df.groupby('Season')['Modal_Price'].agg(['mean', 'std', 'count'])
print('Seasonal Price Statistics:')
print(seasonal_price)

plt.figure(figsize=(10, 6))
df.boxplot(column='Modal_Price', by='Season', figsize=(10, 6))
plt.title('Modal Price Distribution by Season')
plt.suptitle('')
plt.xlabel('Season')
plt.ylabel('Modal Price (Rs./Quintal)')
plt.tight_layout()
plt.show()

## 6. Chi-Square Test: Season vs Crop Independence

Testing whether the distribution of crops across seasons is statistically independent.


In [ ]:
from scipy.stats import chi2_contingency

contingency = pd.crosstab(df['Season'], df['Crop'])
chi2, p_value, dof, expected = chi2_contingency(contingency)

print('Chi-Square Test of Independence: Season vs Crop')
print(f'Chi-square statistic: {chi2:.4f}')
print(f'p-value: {p_value:.6f}')
print(f'Degrees of freedom: {dof}')
print(f'\nInterpretation: ', end='')
if p_value < 0.05:
    print('Reject null hypothesis - Season and Crop are NOT independent (statistically significant association)')
else:
    print('Fail to reject null hypothesis - Season and Crop appear independent')

## 7. Model Performance: Random Forest for Yield Prediction

The Random Forest Regressor estimates crop yield using features like area, production, and modal price. Feature importance identifies which variables most strongly drive yield.


In [ ]:
from data_pipeline import train_random_forest_model

model_metrics, feature_importances = train_random_forest_model(df)
print('\nModel Performance Metrics:')
print(f"RMSE: {model_metrics['RMSE']:.4f}")
print(f"R² Score: {model_metrics['R2']:.4f}")

plt.figure(figsize=(10, 6))
feature_importances.head(12).sort_values(ascending=True).plot(kind='barh', color='seagreen')
plt.title('Top 12 Feature Importances for Yield Prediction', fontweight='bold')
plt.xlabel('Importance Score')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

## 8. Cleaned Dataset Summary

The final cleaned and engineered dataset, ready for Tableau dashboard integration.


In [ ]:
print('Final Cleaned Dataset:')
print(f'Shape: {df.shape}')
print(f'\nColumns: {list(df.columns)}')
print(f'\nData Types:\n{df.dtypes}')
print(f'\nSample rows:\n')
df.head(10)

## 9. Export Summary Statistics for Reporting

Key statistics for inclusion in the project report and data story.


In [ ]:
summary_stats = {
    'Total Records': len(df),
    'Unique Crops': df['Crop'].nunique(),
    'Unique Seasons': df['Season'].nunique(),
    'Date Range': f"{df.get('Year', pd.Series()).min() if 'Year' in df.columns else 'N/A'} to {df.get('Year', pd.Series()).max() if 'Year' in df.columns else 'N/A'}",
    'Avg Yield': f"{df['Yield'].mean():.2f}",
    'Avg Modal Price': f"₹{df['Modal_Price'].mean():.2f}",
    'Avg Estimated Revenue': f"₹{df['Estimated_Revenue_INR'].mean():,.0f}",
}

print('\n=== KEY PROJECT STATISTICS ===\n')
for key, value in summary_stats.items():
    print(f'{key}: {value}')

print(f'\nDataset saved to: data/cleaned_agriculture_data.csv')
print('\nReady for Tableau dashboard integration!')

## Conclusion

The data pipeline and notebook together support evidence-based analysis for agricultural market operations. The cleaned and engineered dataset is now ready for deployment in Tableau dashboards and business intelligence workflows, enabling stakeholders to identify crop patterns, seasonal behavior, revenue signals, and yield drivers in real time.
